# Yambda — Kaggle setup & smoke test

Notebook-driven workflow: install the `ymrec` package from GitHub, then reproduce **Milestone 0** (MostPop on Yambda-50M) to confirm the Kaggle environment works end-to-end.

**Before running** — in the right-hand panel:
- *Settings → Internet*: **On** (needed to pip-install and download the dataset).
- GPU not needed for this notebook (CPU is fine).
- *Optional*: *Add-ons → Secrets* → add a secret named `HF_TOKEN` (your HuggingFace read token). Anonymous download also works, just slower / rate-limited.

In [ ]:
# Install the project package straight from GitHub (thin notebook, versioned code).
!pip install -q "git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
# Load the HF token from Kaggle Secrets if present (optional).
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secret.")
except Exception as e:
    print("No HF_TOKEN secret — anonymous download still works.", e)

## Milestone 0 — MostPop on Yambda-50M

Downloads `listens.parquet` (~369 MB) on first run, applies the Global Temporal Split, and scores the most-popular-track baseline. Expected: NDCG@10 ≈ 0.017–0.019 (paper: 0.0186).

In [ ]:
import polars as pl
from ymrec.config import LISTEN_POSITIVE_RATIO, TOPK, PAPER_BASELINES_NDCG10
from ymrec.data.yambda import interactions_path
from ymrec.eval import split as gts
from ymrec.eval.metrics import evaluate_ranking
from ymrec.baselines.most_pop import popularity_ranking, recommend

K = max(TOPK)

path = interactions_path("listens", size="50m", layout="flat")
listens = pl.read_parquet(path, columns=["uid", "item_id", "timestamp", "played_ratio_pct"])
pos = listens.filter(pl.col("played_ratio_pct") >= LISTEN_POSITIVE_RATIO)
print(f"raw listens={listens.height:,}  Listen+={pos.height:,}")

train, test, bounds = gts.split(pos)
print(f"train={train.height:,}  test={test.height:,}  bounds={bounds}")

In [ ]:
def per_user_sets(df):
    agg = df.group_by("uid").agg(pl.col("item_id").unique().alias("items"))
    return {int(u): set(it) for u, it in zip(agg["uid"], agg["items"])}

train_seen = per_user_sets(train)
test_rel = per_user_sets(test)
users = sorted(set(test_rel) & set(train_seen))
relevant = [test_rel[u] for u in users]
print(f"evaluated users: {len(users):,}")

ranking = popularity_ranking(train)
recs = recommend(ranking, users, K)  # MostPop: do NOT filter seen items (re-listening is normal)
res = evaluate_ranking(recs, relevant, n_items=pos["item_id"].n_unique(), ks=TOPK)

print(f"MostPop NDCG@10 = {res['ndcg@10']:.4f}  (paper {PAPER_BASELINES_NDCG10['50m']['MostPop']})")
res